In [ ]:
import pandas as pd

# This is the "Raw" link to the Johns Hopkins global confirmed cases
url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"

# We use pandas (pd) to read the data
df = pd.read_csv(url)

# This command shows us the first 5 rows of the data
df.head()

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,NaN,Afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,NaN,Albania,41.15330,20.168300,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,NaN,Algeria,28.03390,1.659600,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,NaN,Andorra,42.50630,1.521800,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,NaN,Angola,-11.20270,17.873900,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288


In [ ]:
# We tell Python: "Keep the location info, but squash all the date columns into one"
df_melted = df.melt(
    id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'],
    var_name='Date',
    value_name='Confirmed_Cases'
)

# We tell Python that the "Date" column contains actual calendar dates
df_melted['Date'] = pd.to_datetime(df_melted['Date'])

# Now let's see how it looks!
df_melted.head()

/tmp/ipykernel_7146/4236965497.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_melted['Date'] = pd.to_datetime(df_melted['Date'])


,Province/State,Country/Region,Lat,Long,Date,Confirmed_Cases
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0
1,NaN,Albania,41.15330,20.168300,2020-01-22,0
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0


In [ ]:
# Sort the data by Country and Date to make sure the math is correct
df_melted = df_melted.sort_values(by=['Country/Region', 'Date'])

# Create a 'New_Cases' column by finding the difference between today and yesterday
# we group by country so we don't accidentally subtract Afghanistan's cases from Albania's!
df_melted['New_Cases'] = df_melted.groupby('Country/Region')['Confirmed_Cases'].diff().fillna(0)

# Sometimes reporting errors cause negative numbers (ignore them)
df_melted['New_Cases'] = df_melted['New_Cases'].clip(lower=0)

df_melted.head()

,Province/State,Country/Region,Lat,Long,Date,Confirmed_Cases,New_Cases
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0,0.0
289,NaN,Afghanistan,33.93911,67.709953,2020-01-23,0,0.0
578,NaN,Afghanistan,33.93911,67.709953,2020-01-24,0,0.0
867,NaN,Afghanistan,33.93911,67.709953,2020-01-25,0,0.0
1156,NaN,Afghanistan,33.93911,67.709953,2020-01-26,0,0.0


In [ ]:
# Sum up all provinces into one total for each country
df_country = df_melted.groupby(['Country/Region', 'Date'])[['Confirmed_Cases', 'New_Cases']].sum().reset_index()

# Check how many rows we have now
print(f"Total rows: {len(df_country)}")
df_country.tail()

Total rows: 229743


,Country/Region,Date,Confirmed_Cases,New_Cases
229738,Zimbabwe,2023-03-05,264127,0.0
229739,Zimbabwe,2023-03-06,264127,0.0
229740,Zimbabwe,2023-03-07,264127,0.0
229741,Zimbabwe,2023-03-08,264276,149.0
229742,Zimbabwe,2023-03-09,264276,0.0


In [ ]:
# The official Google Mobility URL
mobility_url = "https://www.gstatic.com/covid19/mobility/Global_Mobility_Report.csv"

# List of columns we actually want (this saves memory)
useful_cols = [
    'country_region', 'sub_region_1', 'date',
    'retail_and_recreation_percent_change_from_baseline',
    'transit_stations_percent_change_from_baseline',
    'workplaces_percent_change_from_baseline',
    'residential_percent_change_from_baseline'
]

# Load the data - this might take a minute!
df_mobility = pd.read_csv(mobility_url, usecols=useful_cols, low_memory=False)

# Convert the date column to match your JHU data format
df_mobility['date'] = pd.to_datetime(df_mobility['date'])

df_mobility.head()

,country_region,sub_region_1,date,retail_and_recreation_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
0,United Arab Emirates,NaN,2020-02-15,0.0,0.0,2.0,1.0
1,United Arab Emirates,NaN,2020-02-16,1.0,1.0,2.0,1.0
2,United Arab Emirates,NaN,2020-02-17,-1.0,1.0,2.0,1.0
3,United Arab Emirates,NaN,2020-02-18,-2.0,0.0,2.0,1.0
4,United Arab Emirates,NaN,2020-02-19,-2.0,-1.0,2.0,1.0


In [ ]:
# We only want rows where 'sub_region_1' is empty (this means it's the Country total)
df_mobility_national = df_mobility[df_mobility['sub_region_1'].isna()].copy()

# Rename columns so they match your JHU table perfectly
df_mobility_national = df_mobility_national.rename(columns={
    'country_region': 'Country/Region',
    'date': 'Date'
})

# Drop the empty sub_region column now that we've used it
df_mobility_national = df_mobility_national.drop(columns=['sub_region_1'])

df_mobility_national.head()

,Country/Region,Date,retail_and_recreation_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
0,United Arab Emirates,2020-02-15,0.0,0.0,2.0,1.0
1,United Arab Emirates,2020-02-16,1.0,1.0,2.0,1.0
2,United Arab Emirates,2020-02-17,-1.0,1.0,2.0,1.0
3,United Arab Emirates,2020-02-18,-2.0,0.0,2.0,1.0
4,United Arab Emirates,2020-02-19,-2.0,-1.0,2.0,1.0


In [ ]:
# Combine both tables using 'Date' and 'Country/Region' as the key
df_master = pd.merge(df_country, df_mobility_national, on=['Country/Region', 'Date'], how='inner')

# Show the final result!
df_master.tail()

,Country/Region,Date,Confirmed_Cases,New_Cases,retail_and_recreation_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
180311,Zimbabwe,2022-10-11,257749,0.0,122.0,142.0,101.0,13.0
180312,Zimbabwe,2022-10-12,257798,49.0,123.0,148.0,100.0,13.0
180313,Zimbabwe,2022-10-13,257827,29.0,130.0,149.0,94.0,13.0
180314,Zimbabwe,2022-10-14,257827,0.0,132.0,149.0,100.0,11.0
180315,Zimbabwe,2022-10-15,257827,0.0,135.0,163.0,120.0,9.0


In [ ]:
# This creates the file inside Colab's temporary memory
df_master.to_csv('master_epidemic_data.csv', index=False)

In [ ]:
# The official raw URL for OWID vaccination data
vax_url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/vaccinations.csv"

# Load the data
df_vax = pd.read_csv(vax_url)

# We only need the Date, Location, and the percentage of people vaccinated
vax_cols = ['location', 'date', 'people_fully_vaccinated_per_hundred', 'total_boosters_per_hundred']
df_vax_clean = df_vax[vax_cols].copy()

# Rename to match your Master Table
df_vax_clean = df_vax_clean.rename(columns={'location': 'Country/Region', 'date': 'Date'})
df_vax_clean['Date'] = pd.to_datetime(df_vax_clean['Date'])

# Merge it into your existing master data
# Assuming your previous table is named 'df_master'
df_master = pd.merge(df_master, df_vax_clean, on=['Country/Region', 'Date'], how='left')

# Fill missing vaccination values with the previous day (since vax rates don't drop)
df_master = df_master.sort_values(by=['Country/Region', 'Date'])
df_master[['people_fully_vaccinated_per_hundred', 'total_boosters_per_hundred']] = df_master.groupby('Country/Region')[['people_fully_vaccinated_per_hundred', 'total_boosters_per_hundred']].ffill().fillna(0)

df_master.tail()

,Country/Region,Date,Confirmed_Cases,New_Cases,retail_and_recreation_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline,people_fully_vaccinated_per_hundred,total_boosters_per_hundred
180311,Zimbabwe,2022-10-11,257749,0.0,122.0,142.0,101.0,13.0,29.11,6.33
180312,Zimbabwe,2022-10-12,257798,49.0,123.0,148.0,100.0,13.0,29.11,6.33
180313,Zimbabwe,2022-10-13,257827,29.0,130.0,149.0,94.0,13.0,29.11,6.33
180314,Zimbabwe,2022-10-14,257827,0.0,132.0,149.0,100.0,11.0,29.11,6.33
180315,Zimbabwe,2022-10-15,257827,0.0,135.0,163.0,120.0,9.0,29.11,6.33


In [ ]:
# We will create 7-day and 14-day lags for New Cases and Mobility
cols_to_lag = [
    'New_Cases',
    'retail_and_recreation_percent_change_from_baseline',
    'workplaces_percent_change_from_baseline'
]

for col in cols_to_lag:
    # 7-day lag (What happened a week ago?)
    df_master[f'{col}_lag_7'] = df_master.groupby('Country/Region')[col].shift(7)

    # 14-day lag (What happened two weeks ago?)
    df_master[f'{col}_lag_14'] = df_master.groupby('Country/Region')[col].shift(14)

# Remove the rows where lags are empty (the first 14 days of the dataset)
df_final = df_master.dropna().copy()

print("New columns added:", [c for c in df_final.columns if 'lag' in c])
df_final.head()

New columns added: ['New_Cases_lag_7', 'New_Cases_lag_14', 'retail_and_recreation_percent_change_from_baseline_lag_7', 'retail_and_recreation_percent_change_from_baseline_lag_14', 'workplaces_percent_change_from_baseline_lag_7', 'workplaces_percent_change_from_baseline_lag_14']


,Country/Region,Date,Confirmed_Cases,New_Cases,retail_and_recreation_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline,people_fully_vaccinated_per_hundred,total_boosters_per_hundred,New_Cases_lag_7,New_Cases_lag_14,retail_and_recreation_percent_change_from_baseline_lag_7,retail_and_recreation_percent_change_from_baseline_lag_14,workplaces_percent_change_from_baseline_lag_7,workplaces_percent_change_from_baseline_lag_14
14,Afghanistan,2020-02-22,0,0.0,2.0,7.0,6.0,0.0,0.0,0.0,0.0,0.0,-1.0,-9.0,4.0,-28.0
15,Afghanistan,2020-02-22,0,0.0,-2.0,5.0,4.0,0.0,0.0,0.0,0.0,0.0,-1.0,-13.0,5.0,-39.0
16,Afghanistan,2020-02-23,0,0.0,2.0,7.0,6.0,1.0,0.0,0.0,0.0,0.0,-4.0,3.0,1.0,4.0
17,Afghanistan,2020-02-23,0,0.0,-1.0,5.0,5.0,1.0,0.0,0.0,0.0,0.0,-2.0,0.0,6.0,2.0
18,Afghanistan,2020-02-24,5,5.0,3.0,9.0,7.0,0.0,0.0,0.0,0.0,0.0,-3.0,6.0,5.0,5.0


In [ ]:
# Save the final file
df_final.to_csv('ultra_epidemic_data.csv', index=False)

In [ ]:
# This gives a statistical summary of your final dataset
summary = df_final.describe()

# Save it as a text file for your teammate
summary.to_csv('data_summary.csv')

print("Summary statistics generated. Share this with your ML Developer!")

Summary statistics generated. Share this with your ML Developer!


In [ ]:
# Create a 7-day moving average for New_Cases
df_final['New_Cases_7Day_Avg'] = df_final.groupby('Country/Region')['New_Cases'].transform(lambda x: x.rolling(window=7).mean())

# Fill the first 6 days of the average (which would be NaN) with the actual New_Cases
df_final['New_Cases_7Day_Avg'] = df_final['New_Cases_7Day_Avg'].fillna(df_final['New_Cases'])

print("Smoothing complete!")

Smoothing complete!


In [ ]:
# Check if there are any missing values left in the whole table
nan_counts = df_final.isnull().sum()

print("Missing values per column:")
print(nan_counts)

# If any exist, fill them with 0 as a final safety net
df_final = df_final.fillna(0)

print("\nAudit Complete: All missing values have been handled.")

Missing values per column:
Country/Region                                               0
Date                                                         0
Confirmed_Cases                                              0
New_Cases                                                    0
retail_and_recreation_percent_change_from_baseline           0
transit_stations_percent_change_from_baseline                0
workplaces_percent_change_from_baseline                      0
residential_percent_change_from_baseline                     0
people_fully_vaccinated_per_hundred                          0
total_boosters_per_hundred                                   0
New_Cases_lag_7                                              0
New_Cases_lag_14                                             0
retail_and_recreation_percent_change_from_baseline_lag_7     0
retail_and_recreation_percent_change_from_baseline_lag_14    0
workplaces_percent_change_from_baseline_lag_7                0
workplaces_percent_change_fr

In [ ]:
# Export the absolute final version
df_final.to_csv('final_processed_epidemic_data.csv', index=False)

In [ ]:
# Check the shape and column names
print(f"Final Dataset Shape: {df_final.shape}")
print("\nColumns ready for ML & UI:")
print(df_final.columns.tolist())

# Check for any remaining 'Zero' or 'Null' issues in critical columns
critical_cols = ['New_Cases_7Day_Avg', 'retail_and_recreation_percent_change_from_baseline_lag_14']
print("\nCheck for data quality:")
print(df_final[critical_cols].describe())

Final Dataset Shape: (175154, 17)

Columns ready for ML & UI:
['Country/Region', 'Date', 'Confirmed_Cases', 'New_Cases', 'retail_and_recreation_percent_change_from_baseline', 'transit_stations_percent_change_from_baseline', 'workplaces_percent_change_from_baseline', 'residential_percent_change_from_baseline', 'people_fully_vaccinated_per_hundred', 'total_boosters_per_hundred', 'New_Cases_lag_7', 'New_Cases_lag_14', 'retail_and_recreation_percent_change_from_baseline_lag_7', 'retail_and_recreation_percent_change_from_baseline_lag_14', 'workplaces_percent_change_from_baseline_lag_7', 'workplaces_percent_change_from_baseline_lag_14', 'New_Cases_7Day_Avg']

Check for data quality:
       New_Cases_7Day_Avg  \
count        1.751540e+05   
mean         1.498696e+05   
std          1.544530e+06   
min          0.000000e+00   
25%          5.700000e+01   
50%          5.630714e+02   
75%          3.425536e+03   
max          3.561801e+07   

       retail_and_recreation_percent_change_from_bas